# 01 - Data Collection

**Source:** [Bandcamp Daily, *Album of the Day*](https://daily.bandcamp.com/album-of-the-day)
**Output:** one row per published article -> `data/interim/scraped_articles.csv`

Bandcamp Daily has published an Album of the Day almost every weekday since
2011. There is no API, so the archive has to be scraped. This notebook walks
through how, and why the crawler is built the way it is.

The scraping logic itself lives in `src/bandcamp_aotd/extract/` and is
imported here rather than defined inline. That is deliberate: the scheduled
refresh runs the exact same code this notebook demonstrates, so there is no
second copy to drift out of sync.

| Stage | Module | What it does |
|---|---|---|
| Discover | `extract/discover.py` | Walk the paginated index, collect article URLs |
| Fetch | `extract/fetch.py` | Download each article once, cache it on disk |
| Parse | `extract/parse.py` | Pull seven fields out of each page |

> **Running this end to end takes ~40 minutes on a cold cache** (2,300 articles
> at ~1 request/second). Everything below is safe to run repeatedly - the cache
> means the second pass is instant.

In [ ]:
import sys
from pathlib import Path

# Make the pipeline package importable without installing it, so the notebook
# runs on a fresh clone. `pip install -e .` also works and is preferred.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bandcamp_aotd.logging_config import setup_logging

setup_logging()

## 1. Discover article URLs

The archive is paginated 30 articles per page, newest first. Two design
decisions matter here:

**Politeness.** Bandcamp Daily is a small editorial site, not an API. The
session sends a real User-Agent, warms up cookies before paginating, sleeps
between requests, and honours `Retry-After` on a 429. A scraper that gets a
project's IP banned is not a working scraper.

**Incrementality.** Because the index is newest-first, a refresh only needs to
read until it starts seeing articles it already has. Passing `known_urls`
turns a 500-page crawl into a 1-page one — the difference between a weekly job
that costs one HTTP request and one that costs five hundred.

In [ ]:
from bandcamp_aotd.extract import build_session, discover_article_urls

session = build_session()

# max_pages=2 keeps this demo to ~60 articles. Drop it for a full backfill.
urls = discover_article_urls(session, max_pages=2)

print(f"Found {len(urls)} article URLs")
urls[:3]

### What an incremental refresh looks like

In production the pipeline hands `discover_article_urls` the URLs already in
the warehouse. The crawler stops after seeing one full page of familiar
articles — enough to be confident it has caught up without betting the whole
run on a single URL.

In [ ]:
from bandcamp_aotd.load import fetch_known_article_urls

# Reads from Postgres; returns an empty set (and logs a warning) if the
# database isn't configured yet, so this cell is safe to run either way.
known = fetch_known_article_urls()
print(f"{len(known)} articles already in the warehouse")

## 2. Fetch article pages

Every page is written to `data/raw/html_cache/<slug>.html` the first time it
is downloaded and read from disk afterwards.

This is the single most useful decision in the extract layer. Regexes against
scraped HTML *will* need fixing. Without a cache, every fix means re-crawling
2,300 pages; with one, re-parsing the whole archive takes seconds and can be
done offline, on a plane, with no risk of being rate limited.

In [ ]:
from bandcamp_aotd.extract import fetch_articles

pages = list(fetch_articles(session, urls[:5]))
url, html = pages[0]

print(url)
print(f"{len(html):,} characters of HTML")

## 3. Parse each page

Seven fields come out of each article:

| Field | Where it comes from |
|---|---|
| `published_date` | the byline |
| `title` | `<title>`, minus site branding and the "Album of the Day:" prefix |
| `artist` / `album` | the title, split on the first comma |
| `genre_tag` | the `article:tag` meta property |
| `record_label` | the album sidebar |
| `label_location_raw` | the label's self-reported location |
| `author` | the contributor byline |

Each extractor has a **primary regex** — the expression that has been pulling
this dataset correctly since 2021 — and a **meta-tag fallback**. If Bandcamp
reskins the site the regex goes quiet and the Open Graph path keeps the
pipeline alive. Keeping the regex first means historical output stays
byte-identical.

Nothing raises. A field that can't be found comes back `None` and the row
records why in `parse_status`, so one odd article can't kill a 2,300-page
crawl.

In [ ]:
from bandcamp_aotd.extract import parse_article

record = parse_article(html, url)
record

### The two business rules worth knowing about

Bandcamp headlines are formatted `Artist, "Album"` — but not always, and the
exceptions carry real meaning:

1. **No comma in the headline** (self-titled records, compilations) leaves
   artist and album identical. In that case the sidebar's label field is
   actually the artist's name, so it gets promoted.
2. **Artist and label are the same entity** means the record is self-released.
   It's stored as `Independent Artist`.

Rule 2 produces the `is_independent` flag, which turns out to be the most
interesting single dimension in the dataset — about two thirds of everything
Bandcamp Daily features is self-released.

In [ ]:
from bandcamp_aotd.extract.parse import resolve_label_and_artist

# Rule 1: headline had no comma
print(resolve_label_and_artist("Untitled", "Untitled", "Boomkat"))

# Rule 2: artist is their own label
print(resolve_label_and_artist("Aphex Twin", "Syro", "Aphex Twin"))

## 4. Run the full scrape

`scrape_articles` chains discover -> fetch -> parse and returns a DataFrame.

From the command line the same thing is:

```bash
python -m bandcamp_aotd scrape              # incremental
python -m bandcamp_aotd scrape --full       # full backfill
```

In [ ]:
from bandcamp_aotd.extract import scrape_articles

# Raise or remove max_pages for a real run.
df = scrape_articles(max_pages=2)
print(df.shape)
df.head()

In [ ]:
df["parse_status"].value_counts()

## Next

`02_spotify_enrichment.ipynb` attaches Spotify catalog metadata — cover art,
release dates and links — to what was scraped here.